In [8]:
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
import re

**1. Построй динамику показателя NPS Качества связи в агрегации по месяцам**

In [9]:
def find_optimal_pattern_stepwise(df, min_pattern_length=3, max_depth=15):

    numbers_series = df['ANUMBER_NORM'].astype(str)
    total_numbers = len(df)
    current_pattern = ""
    best_pattern = ""
    max_coverage = 0
    pattern_history = []
    
    for depth in range(1, max_depth + 1):
        next_digit_stats = defaultdict(int)
        
        for number in numbers_series:
            if len(number) >= depth and number.startswith(current_pattern):
                if len(number) >= depth:
                    next_digit = number[depth-1:depth]
                    next_digit_stats[next_digit] += 1
        
        if not next_digit_stats:
            break
            
        most_common_digit, count = max(next_digit_stats.items(), key=lambda x: x[1])
        new_pattern = current_pattern + most_common_digit
        coverage_percent = (count / total_numbers) * 100
        
        pattern_history.append({
            'pattern': new_pattern,
            'coverage': count,
            'coverage_percent': coverage_percent,
            'depth': depth
        })
        
        print(f"Глубина {depth:2d}: {new_pattern:<15} - {count:5d} номеров ({coverage_percent:5.1f}%)")
        
        if depth >= min_pattern_length and count > max_coverage:
            best_pattern = new_pattern
            max_coverage = count
        
        current_pattern = new_pattern
        if coverage_percent < 1.0:
            print(f"Остановка: охват ниже 1%")
            break
    
    return best_pattern, max_coverage, pattern_history

def find_multiple_optimal_patterns(df, num_patterns=3, min_length=3, max_depth=12):
    numbers_series = df['ANUMBER_NORM'].astype(str)
    all_patterns = []
    initial_prefixes = defaultdict(int)
    for number in numbers_series:
        if len(number) >= 3:
            prefix = number[:3]
            initial_prefixes[prefix] += 1
    
    top_initial = sorted(initial_prefixes.items(), key=lambda x: x[1], reverse=True)[:num_patterns]
    
    for i, (initial_prefix, initial_count) in enumerate(top_initial, 1):
        print(f"\Анализ паттерна {i} на основе '{initial_prefix}':")
        print("-" * 35)
        
        current_pattern = initial_prefix
        current_coverage = initial_count
        
        for depth in range(4, max_depth + 1):
            next_digit_stats = defaultdict(int)
            
            for number in numbers_series:
                if len(number) >= depth and number.startswith(current_pattern):
                    next_digit = number[depth-1:depth]
                    next_digit_stats[next_digit] += 1
            
            if not next_digit_stats:
                break
                
            most_common_digit, count = max(next_digit_stats.items(), key=lambda x: x[1])
            
            if count < current_coverage * 0.3:
                print(f"  Остановка: охват упал с {current_coverage} до {count}")
                break
                
            current_pattern += most_common_digit
            current_coverage = count
            
            coverage_percent = (current_coverage / len(df)) * 100
            print(f"  Глубина {depth}: {current_pattern} - {current_coverage} номеров ({coverage_percent:.1f}%)")
        
        all_patterns.append((current_pattern, current_coverage))
    
    return all_patterns

def analyze_pattern_distribution(df, pattern):
    numbers_series = df['ANUMBER_NORM'].astype(str)
    mask = numbers_series.str.startswith(pattern)
    pattern_numbers = df[mask]
    
    if len(pattern_numbers) == 0:
        return {}
    
    next_digit_stats = defaultdict(int)
    for number in numbers_series[mask]:
        if len(number) > len(pattern):
            next_digit = number[len(pattern):len(pattern)+1]
            next_digit_stats[next_digit] += 1
    
    return dict(sorted(next_digit_stats.items(), key=lambda x: x[1], reverse=True))

def get_top_prefixes_enhanced(df, top_n=5, min_length=3, max_length=6):
    numbers_series = df['ANUMBER_NORM'].astype(str)
    prefix_stats = defaultdict(int)
    
    for number in numbers_series:
        for length in range(min_length, min(max_length + 1, len(number) + 1)):
            prefix = number[:length]
            prefix_stats[prefix] += 1
    
    significant_prefixes = {prefix: count for prefix, count in prefix_stats.items() 
                          if count >= 2}
    
    return sorted(significant_prefixes.items(), key=lambda x: x[1], reverse=True)[:top_n]

df = pd.read_excel('C://Users//molot//Downloads//Flash call.xlsx', engine='openpyxl')
best_pattern, max_coverage, history = find_optimal_pattern_stepwise(df)

print(f"\nЧаще всего встречается паттерн: {best_pattern}")
print(f"Охват: {max_coverage} номеров из {len(df)} ({max_coverage/len(df)*100:.1f}%)")

mask = df['ANUMBER_NORM'].astype(str).str.startswith(best_pattern)
best_numbers = df[mask]
total_calls = best_numbers['CALL_CNT'].sum()

print(f"Кол-во звонков: {total_calls}")
print(f"Средняя активность: {total_calls/max_coverage:.1f} звонков на номер")
print(f"\nАнализ паттерна '{best_pattern}':")
pattern_stats = analyze_pattern_distribution(df, best_pattern)
for digit, count in list(pattern_stats.items())[:5]:
    percent = (count / max_coverage) * 100
    print(f"  {best_pattern}{digit}* - {count:4d} номеров ({percent:5.1f}% от паттерна)")

print(f"\nТоп паттернов:")
multiple_patterns = find_multiple_optimal_patterns(df, num_patterns=3)

print(f"\nИтог:")
for i, (pattern, coverage) in enumerate(multiple_patterns, 1):
    percent = (coverage / len(df)) * 100
    print(f"{i}. {pattern:<15} - {coverage:5d} номеров ({percent:5.1f}%)")
print(f"\nДругие паттерны:")
top_prefixes = get_top_prefixes_enhanced(df)

for prefix, count in top_prefixes:
    percent = count / len(df) * 100
    print(f"  {prefix}* - {count} номеров ({percent:.1f}%)")

print(f"\nДополнительная статистика:")
print(f"Всего уникальных номеров: {len(df)}")
print(f"Общее количество звонков: {df['CALL_CNT'].sum()}")
print(f"Средняя активность по всем номерам: {df['CALL_CNT'].sum()/len(df):.1f} звонков на номер")

Глубина  1: 4               -  2945 номеров ( 62.4%)
Глубина  2: 49              -  1532 номеров ( 32.4%)
Глубина  3: 492             -   289 номеров (  6.1%)
Глубина  4: 4927            -    39 номеров (  0.8%)
Остановка: охват ниже 1%

Чаще всего встречается паттерн: 492
Охват: 289 номеров из 4722 (6.1%)
Кол-во звонков: 294
Средняя активность: 1.0 звонков на номер

Анализ паттерна '492':
  4927* -   39 номеров ( 13.5% от паттерна)
  4928* -   38 номеров ( 13.1% от паттерна)
  4921* -   35 номеров ( 12.1% от паттерна)
  4925* -   35 номеров ( 12.1% от паттерна)
  4922* -   32 номеров ( 11.1% от паттерна)

Топ паттернов:
\Анализ паттерна 1 на основе '336':
-----------------------------------
  Остановка: охват упал с 307 до 40
\Анализ паттерна 2 на основе '492':
-----------------------------------
  Остановка: охват упал с 289 до 39
\Анализ паттерна 3 на основе '493':
-----------------------------------
  Остановка: охват упал с 286 до 37

Итог:
1. 336             -   307 номеров (  6.

c:\Users\molot\AppData\Local\Programs\Python\Python314\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


In [10]:
def analyze_patterns(df):
    print("Анализ паттернов")
    total_len = len(df)
    total_calls = df['CALL_CNT'].sum()
    
    print(f"Всего номеров: {total_len}")
    print(f"Всего звонков: {total_calls}")
    print(f"Средняя активность: {total_calls/total_len:.2f} звонков/номер")
    df['number_length'] = df['ANUMBER_NORM'].astype(str).str.len()
    length_stats = df['number_length'].value_counts().sort_index()
    return length_stats

def detect_characteristics(df):
    
    numbers_series = df['ANUMBER_NORM'].astype(str)
    flash_candidates = []
    single_call_mask = df['CALL_CNT'] == 1
    single_call_numbers = df[single_call_mask]
    
    print(f"\nНомера с 1 звонком: {len(single_call_numbers)}")
    
    prefix_lengths = [3, 4, 5, 6]
    prefix_patterns = {}
    
    for length in prefix_lengths:
        prefixes = numbers_series.str[:length]
        prefix_counts = prefixes.value_counts()
        significant_prefixes = []
        for prefix, count in prefix_counts.head(20).items():
            prefix_mask = numbers_series.str.startswith(prefix)
            prefix_single_calls = len(df[prefix_mask & single_call_mask])
            single_call_ratio = prefix_single_calls / count if count > 0 else 0
            
            if count >= 10 and single_call_ratio >= 0.7:
                significant_prefixes.append({
                    'prefix': prefix,
                    'total_numbers': count,
                    'single_calls': prefix_single_calls,
                    'single_ratio': single_call_ratio,
                    'avg_calls': df[prefix_mask]['CALL_CNT'].mean()
                })
        
        prefix_patterns[length] = significant_prefixes
    
    return prefix_patterns, single_call_numbers

def dop_pattern_analyse(df, phone_column='ANUMBER_NORM'):

    numbers = df[phone_column].astype(str)
    print(f"\n Основной анализ")
    
    def find_repeating_patterns(number):
        repeats = re.findall(r'(\d)\1{2,}', number)
        return len(repeats) > 0
    
    def find_sequences(number):
        sequences = re.findall(r'(012|123|234|345|456|567|678|789|987|876|765|654|543|432|321|210)', number)
        return len(sequences) > 0
    
    df['has_repeats'] = numbers.apply(find_repeating_patterns)
    df['has_sequences'] = numbers.apply(find_sequences)
    
    repeat_stats = df['has_repeats'].value_counts()
    sequence_stats = df['has_sequences'].value_counts()
    
    print(f"Номера с повторяющимися цифрами: {repeat_stats.get(True, 0)}")
    print(f"Номера с идущими подряд цифрами: {sequence_stats.get(True, 0)}")
    
    return df

def generate_blocking_patterns(prefix_patterns, single_call_numbers):
    print(f"\nПаттерны номеров Flash Call")
    blocking_patterns = []
    
    for length, patterns in prefix_patterns.items():
        for pattern in patterns[:10]:
            if pattern['single_ratio'] >= 0.8:
                blocking_patterns.append({
                    'pattern': f"{pattern['prefix']}*",
                    'coverage': pattern['total_numbers'],
                    'single_call_ratio': pattern['single_ratio'],
                    'avg_calls': pattern['avg_calls'],
                    'type': f'prefix_{length}',
                    'confidence': 'high' if pattern['single_ratio'] > 0.9 else 'medium'
                })
    blocking_patterns.sort(key=lambda x: x['coverage'], reverse=True)
    
    for i, pattern in enumerate(blocking_patterns[:20], 1):
        single_ratio_percent = pattern['single_call_ratio'] * 100
        print(f"{i:2d}. {pattern['pattern']:<5} - {pattern['coverage']:4d} номеров "
            f"(1 звонок: {single_ratio_percent:.1f}%)")
    return blocking_patterns 

def main(file_path):
    try:
        df = pd.read_excel(file_path)
        phone_column = 'ANUMBER_NORM'
        calls_column = 'CALL_CNT'
        
        length_stats = analyze_patterns(df)
        prefix_patterns, single_call_numbers = detect_characteristics(df)
        df = dop_pattern_analyse(df, phone_column)
        blocking_patterns = generate_blocking_patterns(prefix_patterns, single_call_numbers)
        
        return blocking_patterns, single_call_numbers
        
    except Exception as e:
        print(f"Ошибка: {e}")
        return [], pd.DataFrame()
    

if __name__ == "__main__":
    file_path = 'C://Users//molot//Downloads//Flash call.xlsx' 
    patterns, candidates = main(file_path)
    
    if patterns:
        total_coverage = sum(p['coverage'] for p in patterns)
        print(f"\nИтог:")
        print(f"    Найдено паттернов: {len(patterns)}")
        print(f"    Общее покрытие: {total_coverage} номеров")
        print(f"    Количество Flash Call номеров: {len(candidates)}")
    else:
        print("Не найдены паттерны")

c:\Users\molot\AppData\Local\Programs\Python\Python314\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


Анализ паттернов
Всего номеров: 4722
Всего звонков: 5228
Средняя активность: 1.11 звонков/номер

Номера с 1 звонком: 4483

 Основной анализ
Номера с повторяющимися цифрами: 807
Номера с идущими подряд цифрами: 655

Паттерны номеров Flash Call
 1. 336*  -  307 номеров (1 звонок: 94.8%)
 2. 492*  -  289 номеров (1 звонок: 98.3%)
 3. 493*  -  286 номеров (1 звонок: 99.0%)
 4. 337*  -  282 номеров (1 звонок: 96.1%)
 5. 444*  -  272 номеров (1 звонок: 96.3%)
 6. 445*  -  253 номеров (1 звонок: 98.0%)
 7. 494*  -  253 номеров (1 звонок: 98.8%)
 8. 335*  -  249 номеров (1 звонок: 96.4%)
 9. 334*  -  247 номеров (1 звонок: 96.4%)
10. 446*  -  240 номеров (1 звонок: 95.0%)
11. 3512* -   48 номеров (1 звонок: 100.0%)
12. 3327* -   45 номеров (1 звонок: 91.1%)
13. 3358* -   43 номеров (1 звонок: 97.7%)
14. 4455* -   42 номеров (1 звонок: 100.0%)
15. 3374* -   42 номеров (1 звонок: 100.0%)
16. 4448* -   41 номеров (1 звонок: 100.0%)
17. 3365* -   40 номеров (1 звонок: 100.0%)
18. 3344* -   40 номе